<a href="https://colab.research.google.com/github/danisxde/x5_ml/blob/main/x5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install optuna

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from sklearn.preprocessing import LabelEncoder

import xgboost as xgb
import lightgbm as lgb
import optuna

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
df = pd.read_csv('/content/drive/MyDrive/train.csv')

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 206150 entries, 0 to 206149
Data columns (total 19 columns):
 #   Column                                     Non-Null Count   Dtype  
---  ------                                     --------------   -----  
 0   new_id                                     206150 non-null  int64  
 1   Месяц                                      206150 non-null  int64  
 2   Дата открытия, категориальный              206150 non-null  object 
 3   Торговая площадь, категориальный           206150 non-null  object 
 4   Населенный пункт                           206150 non-null  object 
 5   Регион                                     206150 non-null  object 
 6   Численность населения                      206150 non-null  int64  
 7   Количество домохозяйств                    206150 non-null  int64  
 8   Трафик пеший, в час                        206150 non-null  int64  
 9   Трафик авто, в час                         206150 non-null  int64  
 10  Маркетпл

In [7]:
store_counts = df.groupby('new_id')['Месяц'].nunique()
print(f"Магазинов с 10 месяцами: {(store_counts == 10).sum()} из {len(store_counts)}")

Магазинов с 10 месяцами: 20615 из 20615


In [8]:
df[df['Флаг алкогольной лицензии'] == 0]

,new_id,Месяц,"Дата открытия, категориальный","Торговая площадь, категориальный",Населенный пункт,Регион,Численность населения,Количество домохозяйств,"Трафик пеший, в час","Трафик авто, в час","Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м),Количество касс,Флаг алкогольной лицензии,РТО
100,10,1,Средний по возрасту,Средний,Пушкино г,Московская обл,108191,6830,191,212,7,0,0,1,8,2,5,0,27668007.80
101,10,2,Средний по возрасту,Средний,Пушкино г,Московская обл,108191,6830,191,212,7,0,0,1,8,2,5,0,28691671.27
102,10,3,Средний по возрасту,Средний,Пушкино г,Московская обл,108191,6830,191,212,7,0,0,1,8,2,5,0,33355460.71
103,10,4,Средний по возрасту,Средний,Пушкино г,Московская обл,108191,6830,191,212,7,0,0,1,8,2,5,0,30820672.66
104,10,5,Средний по возрасту,Средний,Пушкино г,Московская обл,108191,6830,191,212,7,0,0,1,8,2,5,0,32521965.46
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206025,21312,6,Новый,Средний,Ижевск г,Удмуртская Респ,618128,6866,113,206,4,5,0,7,7,1,8,0,12010430.99
206026,21312,7,Новый,Средний,Ижевск г,Удмуртская Респ,618128,6866,113,206,4,5,0,7,7,1,8,0,13697087.72
206027,21312,8,Новый,Средний,Ижевск г,Удмуртская Респ,618128,6866,113,206,4,5,0,7,7,1,8,0,15089772.38
206028,21312,9,Новый,Средний,Ижевск г,Удмуртская Респ,618128,6866,113,206,4,5,0,7,7,1,8,0,16337214.10


In [9]:
# ---------- 1. Расширенный feature engineering ----------
df = df.sort_values(['new_id', 'Месяц']).copy()

# Лаги (1,2,3) – уже были
for lag in [1,2,3]:
    df[f'RTO_lag_{lag}'] = df.groupby('new_id')['РТО'].shift(lag)

# Дополнительные скользящие статистики
df['RTO_ma_2'] = df.groupby('new_id')['РТО'].transform(lambda x: x.shift(1).rolling(2).mean())
df['RTO_ma_3'] = df.groupby('new_id')['РТО'].transform(lambda x: x.shift(1).rolling(3).mean())
df['RTO_ma_6'] = df.groupby('new_id')['РТО'].transform(lambda x: x.shift(1).rolling(6).mean())  # полгода

# Относительные изменения
df['RTO_growth_1'] = df.groupby('new_id')['РТО'].pct_change(1)
df['RTO_growth_2'] = df.groupby('new_id')['РТО'].pct_change(2)

# Признаки самого магазина (агрегаты за прошлые месяцы)
store_stats = df[df['Месяц'] <= 9].groupby('new_id')['РТО'].agg(['mean', 'median', 'std', 'min', 'max'])
store_stats.columns = ['store_mean_rto', 'store_median_rto', 'store_std_rto', 'store_min_rto', 'store_max_rto']
df = df.merge(store_stats, on='new_id', how='left')

# Отношение текущего РТО к среднему по магазину (для обучающей выборки – сдвиг, чтобы избежать лика)
df['rto_to_store_mean'] = df['РТО'] / (df['store_mean_rto'] + 1e-6)

# Взаимодействия числовых признаков
df['трафик_на_кассу'] = df['Трафик пеший, в час'] / (df['Количество касс'] + 1)
df['население_на_кассу'] = df['Численность населения'] / (df['Количество касс'] + 1)
df['плотность_пятерочек'] = df['Пятерочки (500 м)'] / (df['Продуктовые магазины (500 м)'] + 1)

# Циклические признаки месяца (синус/косинус)
df['month_sin'] = np.sin(2 * np.pi * df['Месяц'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['Месяц'] / 12)

In [10]:
categorical_cols = ['Дата открытия, категориальный', 'Торговая площадь, категориальный',
                    'Населенный пункт', 'Регион', 'Флаг алкогольной лицензии']

def smooth_target_encoding(train_df, cat_col, target_col, prior_weight=20):
    prior = train_df[target_col].mean()
    agg = train_df.groupby(cat_col)[target_col].agg(['mean', 'count'])
    smoothed = (agg['mean'] * agg['count'] + prior * prior_weight) / (agg['count'] + prior_weight)
    return train_df[cat_col].map(smoothed).fillna(prior)

train_part = df[df['Месяц'] <= 9].copy()
global_prior = train_part['РТО'].mean()

for col in categorical_cols:
    encoded_col = f'{col}_te'
    enc_vals = smooth_target_encoding(train_part, col, 'РТО', prior_weight=20)
    train_part[encoded_col] = enc_vals
    mapping = train_part.set_index(col)[encoded_col].to_dict()
    df[encoded_col] = df[col].map(mapping).fillna(global_prior)

In [11]:
numeric_features = ['Численность населения', 'Количество домохозяйств', 'Трафик пеший, в час',
                    'Трафик авто, в час', 'Маркетплейсы, доставки, постаматы (100 м)',
                    'Медицинские уч. и аптеки (300 м)', 'Школы (300 м)', 'Остановки (300 м)',
                    'Продуктовые магазины (500 м)', 'Пятерочки (500 м)', 'Количество касс',
                    'RTO_lag_1', 'RTO_lag_2', 'RTO_lag_3', 'RTO_ma_2', 'RTO_ma_3', 'RTO_ma_6',
                    'RTO_growth_1', 'RTO_growth_2', 'store_mean_rto', 'store_median_rto', 'store_std_rto',
                    'store_min_rto', 'store_max_rto', 'rto_to_store_mean', 'трафик_на_кассу',
                    'население_на_кассу', 'плотность_пятерочек', 'month_sin', 'month_cos']

# Добавляем target encoded колонки
te_cols = [f'{col}_te' for col in categorical_cols]
feature_cols = numeric_features + te_cols + ['Месяц']

# Удаляем строки с NaN (первые несколько месяцев не имеют лагов)
df_clean = df.dropna(subset=['RTO_lag_1', 'RTO_lag_2', 'RTO_lag_3']).copy()
print(f"После удаления NaN: {df_clean.shape}")

После удаления NaN: (144305, 43)


In [12]:
train = df_clean[df_clean['Месяц'] <= 8].copy()
val = df_clean[df_clean['Месяц'] == 9].copy()
future_features = df_clean[df_clean['Месяц'] == 10].copy()  # используем для получения лагов при прогнозе 11

print(f"Train: {train.shape}, Val: {val.shape}, Future features: {future_features.shape}")

X_train = train[feature_cols]
y_train = train['РТО']

X_val = val[feature_cols]
y_val = val['РТО']

Train: (103075, 43), Val: (20615, 43), Future features: (20615, 43)


In [13]:
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1500),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 2.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2.0),
        'eval_metric': 'mape',
        'random_state': 42
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train, verbose=False)
    pred = model.predict(X_val)
    mape = mean_absolute_percentage_error(val['РТО'], pred)
    return mape

# Целевая функция для LightGBM
def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1500),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 2.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2.0),
        'random_state': 42,
        'verbosity': -1
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    mape = mean_absolute_percentage_error(val['РТО'], pred)
    return mape

In [14]:
# Очистка имен колонок от спецсимволов
def clean_column_names(df):
    # Заменяем все символы, кроме букв, цифр и подчеркивания, на подчеркивание
    df.columns = df.columns.str.replace(r'[^\w]', '_', regex=True)
    return df

# Применяем к обучающим и тестовым наборам
X_train = clean_column_names(X_train)
X_val = clean_column_names(X_val)

In [15]:
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=30, show_progress_bar=True)

[I 2026-05-08 13:32:33,329] A new study created in memory with name: no-name-8a773903-fcb1-466e-bc48-6dbf1bc7a2bc


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-05-08 13:33:52,623] Trial 0 finished with value: 0.007002326427696154 and parameters: {'n_estimators': 1206, 'max_depth': 10, 'learning_rate': 0.020022990477176646, 'subsample': 0.9557970627361958, 'colsample_bytree': 0.9648587159453745, 'min_child_weight': 6, 'reg_alpha': 0.9613184112873088, 'reg_lambda': 0.7827840605965906}. Best is trial 0 with value: 0.007002326427696154.
[I 2026-05-08 13:34:26,376] Trial 1 finished with value: 0.008192887569205656 and parameters: {'n_estimators': 1345, 'max_depth': 6, 'learning_rate': 0.020289542014669877, 'subsample': 0.8061151481131765, 'colsample_bytree': 0.6361892437428316, 'min_child_weight': 4, 'reg_alpha': 1.717071490445402, 'reg_lambda': 1.6938828964640769}. Best is trial 0 with value: 0.007002326427696154.
[I 2026-05-08 13:35:24,971] Trial 2 finished with value: 0.006736226490395415 and parameters: {'n_estimators': 874, 'max_depth': 10, 'learning_rate': 0.021118104218400355, 'subsample': 0.7302406443355569, 'colsample_bytree': 0.7

In [16]:
study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=30, show_progress_bar=True)

[I 2026-05-08 14:09:58,012] A new study created in memory with name: no-name-bd776523-c3ed-4e81-b35b-6ec72596ed52


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-05-08 14:10:35,298] Trial 0 finished with value: 0.007595344298858279 and parameters: {'n_estimators': 1169, 'max_depth': 9, 'learning_rate': 0.014524053324116767, 'subsample': 0.6496814387103582, 'colsample_bytree': 0.7010187801611107, 'num_leaves': 229, 'min_child_samples': 26, 'reg_alpha': 1.2599102124548207, 'reg_lambda': 1.7402678581503546}. Best is trial 0 with value: 0.007595344298858279.
[I 2026-05-08 14:11:29,684] Trial 1 finished with value: 0.008182223133119707 and parameters: {'n_estimators': 1480, 'max_depth': 11, 'learning_rate': 0.06587104801943697, 'subsample': 0.8366350717477637, 'colsample_bytree': 0.6356750764760983, 'num_leaves': 212, 'min_child_samples': 12, 'reg_alpha': 1.9663656732908676, 'reg_lambda': 1.4570298060196223}. Best is trial 0 with value: 0.007595344298858279.
[I 2026-05-08 14:12:14,051] Trial 2 finished with value: 0.00908281341182702 and parameters: {'n_estimators': 1140, 'max_depth': 10, 'learning_rate': 0.010937871870598399, 'subsample': 0

In [17]:
print(f"Лучший MAPE XGB: {study_xgb.best_value:.4f}")
print("\nЛучшие параметры XGBoost:", study_xgb.best_params)
print(f"Лучший MAPE LightGBM: {study_lgb.best_value:.4f}")
print(f"Лучшие параметры: {study_lgb.best_params}")

Лучший MAPE XGB: 0.0063

Лучшие параметры XGBoost: {'n_estimators': 1496, 'max_depth': 11, 'learning_rate': 0.02978380382986517, 'subsample': 0.7074096535820478, 'colsample_bytree': 0.7426213358209104, 'min_child_weight': 6, 'reg_alpha': 1.2786833954616519, 'reg_lambda': 0.40904289951311606}
Лучший MAPE LightGBM: 0.0072
Лучшие параметры: {'n_estimators': 1379, 'max_depth': 12, 'learning_rate': 0.0227033362861888, 'subsample': 0.8851196010502747, 'colsample_bytree': 0.8249323930838193, 'num_leaves': 229, 'min_child_samples': 9, 'reg_alpha': 0.8698305267134298, 'reg_lambda': 0.6248071663741475}


In [18]:
train_full = df_clean[df_clean['Месяц'] <= 9].copy()
X_full = train_full[feature_cols]
X_full = clean_column_names(X_full)
y_full = train_full['РТО']

# XGBoost с лучшими параметрами
xgb_params = study_xgb.best_params
xgb_model = xgb.XGBRegressor(**xgb_params, random_state=42)
xgb_model.fit(X_full, y_full)

# LightGBM с лучшими параметрами
lgb_params = study_lgb.best_params
lgb_model = lgb.LGBMRegressor(**lgb_params, random_state=42, verbosity=-1)
lgb_model.fit(X_full, y_full)

LGBMRegressor(colsample_bytree=0.8249323930838193,
              learning_rate=0.0227033362861888, max_depth=12,
              min_child_samples=9, n_estimators=1379, num_leaves=229,
              random_state=42, reg_alpha=0.8698305267134298,
              reg_lambda=0.6248071663741475, subsample=0.8851196010502747,
              verbosity=-1)

In [19]:
# Берём данные месяца 10
month10 = df_clean[df_clean['Месяц'] == 10].copy()
# Для каждого магазина подтянем РТО месяца 9 и 8 из исходного df (не обрезанного)
# Проще: создадим словарь с лагами
store_rto = df[['new_id', 'Месяц', 'РТО']].drop_duplicates().set_index(['new_id', 'Месяц'])['РТО']

def get_lag(row, lag):
    month = row['Месяц'] + lag  # месяц 10 + 1 = 11? нет, нам нужно для месяца 11: лаг1 = РТО месяца 10
    # Для строки месяца 11, лаг1 = РТО_10, лаг2 = РТО_9, лаг3 = РТО_8
    target_month = 11 - lag
    return store_rto.get((row['new_id'], target_month), np.nan)

month11 = month10.copy()
month11['Месяц'] = 11
month11['RTO_lag_1'] = month10.apply(lambda r: get_lag(r, 1), axis=1)   # РТО_10
month11['RTO_lag_2'] = month10.apply(lambda r: get_lag(r, 2), axis=1)   # РТО_9
month11['RTO_lag_3'] = month10.apply(lambda r: get_lag(r, 3), axis=1)   # РТО_8
# Пересчитаем скользящие на основе новых лагов (можно упрощённо)
month11['RTO_ma_2'] = (month11['RTO_lag_1'] + month11['RTO_lag_2']) / 2
month11['RTO_ma_3'] = (month11['RTO_lag_1'] + month11['RTO_lag_2'] + month11['RTO_lag_3']) / 3
month11['RTO_ma_6'] = month11['RTO_ma_3']  # упрощённо, для полноты
month11['RTO_growth_1'] = month11['RTO_lag_1'] / month11['RTO_lag_2'] - 1
month11['RTO_growth_2'] = month11['RTO_lag_1'] / month11['RTO_lag_3'] - 1
month11['rto_to_store_mean'] = month11['RTO_lag_1'] / (month11['store_mean_rto'] + 1e-6)

# Остальные признаки остаются как в month10
X_month11 = month11[feature_cols]
X_month11 = clean_column_names(X_month11)

In [20]:
# Предсказание ансамблем
pred_xgb_log = xgb_model.predict(X_month11)
pred_lgb_log = lgb_model.predict(X_month11)

# Среднее арифметическое в логарифмическом пространстве, затем экспонента
pred_ensemble_log = (pred_xgb_log + pred_lgb_log) / 2
pred_final = pred_ensemble_log

In [21]:
xgb_val = xgb.XGBRegressor(**study_xgb.best_params, random_state=42)
xgb_val.fit(X_train, y_train)
pred_xgb_val = xgb_val.predict(X_val)

lgb_val = lgb.LGBMRegressor(**study_lgb.best_params, random_state=42, verbosity=-1)
lgb_val.fit(X_train, y_train)
pred_lgb_val = lgb_val.predict(X_val)

pred_val_ensemble = (pred_xgb_val + pred_lgb_val) / 2
mape_val = mean_absolute_percentage_error(val['РТО'], pred_val_ensemble)

print(f"\n=== Валидация на месяце 9 ===")
print(f"MAPE XGBoost: {mean_absolute_percentage_error(val['РТО'], pred_xgb_val):.4f}")
print(f"MAPE LightGBM: {mean_absolute_percentage_error(val['РТО'], pred_lgb_val):.4f}")
print(f"MAPE ансамбля: {mape_val:.4f}")


=== Валидация на месяце 9 ===
MAPE XGBoost: 0.0063
MAPE LightGBM: 0.0072
MAPE ансамбля: 0.0061


In [22]:
submission = pd.DataFrame({
    'new_id': month11['new_id'].values,
    'rto': pred_final
})

# Убедимся, что new_id уникальны
submission = submission.drop_duplicates(subset=['new_id'])
print(f"\nСформировано {len(submission)} предсказаний для месяца 11")

# Сохраняем
submission.to_csv('prediction_month11.csv', index=False)
print("Файл prediction_month11.csv сохранён")


Сформировано 20615 предсказаний для месяца 11
Файл prediction_month11.csv сохранён


In [23]:
importance = pd.DataFrame({
    'feature': X_full.columns,
    'importance_gain': xgb_model.feature_importances_
}).sort_values('importance_gain', ascending=False)
print("\nТоп-15 важных признаков (XGBoost):")
print(importance.head(15))


Топ-15 важных признаков (XGBoost):
              feature  importance_gain
11          RTO_lag_1         0.529446
19     store_mean_rto         0.350839
23      store_max_rto         0.055707
14           RTO_ma_2         0.023162
20   store_median_rto         0.012782
24  rto_to_store_mean         0.008285
17       RTO_growth_1         0.008176
18       RTO_growth_2         0.004175
22      store_min_rto         0.000838
21      store_std_rto         0.000629
10    Количество_касс         0.000527
29          month_cos         0.000511
16           RTO_ma_6         0.000426
28          month_sin         0.000425
25    трафик_на_кассу         0.000397
